# Apache Iceberg - Rust

All 34 Rust examples from [docs/iceberg.md](https://platob.github.io/yggdryl/iceberg/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::iceberg::{FormatVersion, PartitionSpec, Table, assign_field_ids};
use yggdryl::local::Folder;
use yggdryl::{arrow, DataType};

use arrow_array::{Int64Array, RecordBatch, StringArray};
use std::sync::Arc;

let mut schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");
assign_field_ids(&mut schema, 1)?;

let path = std::env::temp_dir().join("yggdryl-docs-iceberg-lead");
let _ = std::fs::remove_dir_all(&path);

// A table is created in a folder, and a folder is all it ever touches.
let spec = PartitionSpec::identity(1, &schema, &["venue"])?;
let mut table = Table::create(Folder::new(&path)?, FormatVersion::V2, schema.clone(), spec)?;

// A table that has never been written to has no current snapshot.
assert!(table.current_snapshot().is_none());
assert_eq!(table.scan(None)?.count(), 0);

let batch = RecordBatch::try_new(
    schema.into_arrow_schema()?,
    vec![
        Arc::new(Int64Array::from(vec![1_i64, 2])),
        Arc::new(StringArray::from(vec![Some("XNAS"), Some("XNYS")])),
    ],
)?;
table.commit_append(arrow::batch_reader(batch.schema(), [batch]))?;

let snapshot = table.current_snapshot().expect("a snapshot");
assert_eq!(snapshot.operation(), "append");
assert_eq!(table.data_files()?.len(), 2, "one file per venue");

// Reopening finds the table again, with no catalog in between.
let reopened = Table::open(Folder::new(&path)?)?;
let rows: usize = reopened.scan(None)?.map(|batch| batch.unwrap().num_rows()).sum();
assert_eq!(rows, 2);

## What a table writes

In [ ]:
use yggdryl::iceberg::{FormatVersion, PartitionSpec, Table};
use yggdryl::io::IOBase;
use yggdryl::local::Folder;
use yggdryl::{arrow, DataType};

use arrow_array::{Int64Array, RecordBatch};
use std::sync::Arc;

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");

let path = std::env::temp_dir().join("yggdryl-docs-iceberg-layout");
let _ = std::fs::remove_dir_all(&path);
let mut table = Table::create(
    Folder::new(&path)?,
    FormatVersion::V2,
    schema.clone(),
    PartitionSpec::unpartitioned(),
)?;

let batch = RecordBatch::try_new(
    schema.into_arrow_schema()?,
    vec![Arc::new(Int64Array::from(vec![1_i64]))],
)?;
table.commit_append(arrow::batch_reader(batch.schema(), [batch]))?;

let names: Vec<String> = Folder::new(&path)?
    .ls(true, false)
    .collect::<yggdryl::Result<Vec<_>>>()?
    .iter()
    .filter(|entry| !entry.is_container())
    .filter_map(|entry| entry.url().and_then(|url| url.file_name().map(str::to_owned)))
    .collect();

// One Parquet data file, one manifest, one manifest list, two metadata
// documents (create, then commit), and the version hint that finds them.
assert!(names.iter().any(|name| name.ends_with(".parquet")));
assert!(names.iter().any(|name| name.starts_with("snap-") && name.ends_with(".avro")));
assert!(names.iter().any(|name| name.ends_with("-m0.avro")));
assert!(names.contains(&"v1.metadata.json".to_owned()));
assert!(names.contains(&"v2.metadata.json".to_owned()));
assert!(names.contains(&"version-hint.text".to_owned()));

## Table metadata, v1 through v3

In [ ]:
use yggdryl::iceberg::{FormatVersion, PartitionSpec, TableMetadata};
use yggdryl::DataType;

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");

// v1 keeps the singular `schema` and `partition-spec` keys and has no
// sequence numbers.
let v1 = TableMetadata::new(
    FormatVersion::V1,
    "file:///lake/trades",
    schema.clone(),
    PartitionSpec::unpartitioned(),
)?;
let document = v1.clone().into_json()?;
assert!(document.contains_key("schema"));
assert!(document.contains_key("partition-spec"));
assert!(!document.contains_key("last-sequence-number"));

// v2 makes the plural keys the authority and numbers every commit.
let v2 = TableMetadata::new(
    FormatVersion::V2,
    "file:///lake/trades",
    schema.clone(),
    PartitionSpec::unpartitioned(),
)?;
assert!(v2.clone().into_json()?.contains_key("last-sequence-number"));

// v3 adds row lineage.
let v3 = TableMetadata::new(
    FormatVersion::V3,
    "file:///lake/trades",
    schema,
    PartitionSpec::unpartitioned(),
)?;
assert_eq!(v3.next_row_id(), Some(0));
assert!(v3.clone().into_json()?.contains_key("next-row-id"));

// Every version reads back as itself.
for original in [v1, v2, v3] {
    let read = TableMetadata::from_json(&original.clone().into_json()?)?;
    assert_eq!(read.format_version(), original.format_version());
    assert!(read.current_snapshot().is_none());
}

## Snapshots and the current snapshot

In [ ]:
use yggdryl::iceberg::{FormatVersion, PartitionSpec, TableMetadata};
use yggdryl::DataType;

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");
let metadata = TableMetadata::new(
    FormatVersion::V2,
    "file:///lake/trades",
    schema,
    PartitionSpec::unpartitioned(),
)?;

// A table can have snapshots and still have no current one; that is a
// freshly created table, and a rolled-back one.
assert!(metadata.current_snapshot().is_none());

// `-1` is the other way a document spells "no current snapshot".
let document = metadata.into_json()?.with_key("current-snapshot-id", -1_i64)?;
let read = TableMetadata::from_json(&document)?;
assert!(read.current_snapshot_id().is_none());
assert!(read.current_snapshot().is_none());

## Manifest lists and manifests

In [ ]:
use yggdryl::iceberg::{
    EntryStatus, FormatVersion, PartitionSpec, Table, assign_field_ids, read_manifest,
    read_manifest_spec,
};
use yggdryl::io::IOBase;
use yggdryl::local::Folder;
use yggdryl::{arrow, DataType, MimeType};

use arrow_array::{Int64Array, RecordBatch, StringArray};
use std::sync::Arc;

let mut schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");
assign_field_ids(&mut schema, 1)?;

let path = std::env::temp_dir().join("yggdryl-docs-iceberg-manifests");
let _ = std::fs::remove_dir_all(&path);
let spec = PartitionSpec::identity(1, &schema, &["venue"])?;
let mut table = Table::create(Folder::new(&path)?, FormatVersion::V2, schema.clone(), spec.clone())?;

let batch = RecordBatch::try_new(
    schema.into_arrow_schema()?,
    vec![
        Arc::new(Int64Array::from(vec![1_i64, 2])),
        Arc::new(StringArray::from(vec![Some("XNAS"), Some("XNAS")])),
    ],
)?;
table.commit_append(arrow::batch_reader(batch.schema(), [batch]))?;

// A snapshot names one manifest list; each of its rows is a manifest.
let manifests = table.manifests()?;
assert_eq!(manifests.len(), 1);
assert_eq!(manifests[0].added_files_count, Some(1));
assert_eq!(manifests[0].added_rows_count, Some(2));

// A manifest is self-describing: its Avro header carries the schema and the spec.
let name = manifests[0].manifest_path.rsplit('/').next().unwrap().to_owned();
let handle = Folder::new(&path)?.child_by_path(&format!("metadata/{name}"))?;
assert_eq!(read_manifest_spec(&handle)?, spec);

let entries = read_manifest(&handle)?;
assert_eq!(entries.len(), 1);
assert_eq!(entries[0].status, EntryStatus::Added);
assert_eq!(entries[0].data_file.mime_type, MimeType::PARQUET);
assert_eq!(entries[0].data_file.record_count, 2);

// Statistics are keyed by field id, which is what lets a planner skip a file.
assert!(entries[0].data_file.value_counts.iter().any(|(id, count)| *id == 1 && *count == 2));
assert!(entries[0].data_file.column_sizes.iter().any(|(id, _)| *id == 1));

## Partition specs and the Hive layout

In [ ]:
use yggdryl::iceberg::{PartitionSpec, Transform, assign_field_ids};
use yggdryl::{DataType, Scalar};

let mut schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");
assign_field_ids(&mut schema, 1)?;

let spec = PartitionSpec::identity(1, &schema, &["venue"])?;
assert_eq!(spec.fields[0].source_id, 2);
assert_eq!(spec.fields[0].field_id, 1000);
assert_eq!(spec.fields[0].transform, Transform::Identity);

// The directory chain is the `column=value` shape the crate's Hive reader knows.
assert_eq!(spec.partition_path(&[Scalar::from("XNAS")])?, "venue=XNAS");
assert_eq!(spec.partition_path(&[Scalar::Null])?, "venue=null");

// A partition value is nullable even when its source column is not.
let partition = spec.partition_field(&schema)?;
assert!(partition.fields()[0].is_nullable());

// Invertibility controls restoration, not write support.
assert!(Transform::Identity.is_invertible());
assert!(!Transform::from_str("bucket[16]")?.is_invertible());
assert!(!Transform::Unknown.is_invertible());
assert_eq!(Transform::Bucket(u32::MAX).to_string(), "bucket[4294967295]");
let mut hashed = spec.clone();
hashed.fields[0].name = "venue_bucket".into();
hashed.fields[0].transform = Transform::Bucket(16);
assert!(hashed.require_writable().is_ok());
hashed.fields[0].transform = Transform::Unknown;
assert!(hashed.require_writable().is_err());

In [ ]:
use yggdryl::iceberg::{PartitionSpec, assign_field_ids};
use yggdryl::DataType;

let mut schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");
assign_field_ids(&mut schema, 1)?;
let spec = PartitionSpec::identity(1, &schema, &["venue"])?;

// The tuple describes itself, so the spec reads back off it.
let partition = spec.partition_field(&schema)?;
assert_eq!(partition.iceberg().get("spec-id"), Some("1"));
let venue = partition.get_field_by_name("venue").expect("the partition column");
assert!(venue.is_partition());
assert_eq!(venue.iceberg().get("transform"), Some("identity"));
assert_eq!(PartitionSpec::from_partition_field(&partition)?, spec);

// And a schema that marks its own partition columns needs no column list.
let marked = spec.mark_partitions(&schema)?;
assert_eq!(marked.partition_field_names().collect::<Vec<_>>(), ["venue"]);
assert_eq!(PartitionSpec::from_schema(1, &marked)?, spec);

In [ ]:
use yggdryl::iceberg::{FormatVersion, PartitionSpec, Table, assign_field_ids};
use yggdryl::local::Folder;
use yggdryl::{arrow, DataType};

use arrow_array::{Int64Array, RecordBatch, StringArray};
use std::sync::Arc;

let mut schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");
assign_field_ids(&mut schema, 1)?;

let path = std::env::temp_dir().join("yggdryl-docs-iceberg-null-partition");
let _ = std::fs::remove_dir_all(&path);
let spec = PartitionSpec::identity(1, &schema, &["venue"])?;
let mut table = Table::create(Folder::new(&path)?, FormatVersion::V2, schema.clone(), spec)?;

let batch = RecordBatch::try_new(
    schema.into_arrow_schema()?,
    vec![
        Arc::new(Int64Array::from(vec![1_i64, 2])),
        Arc::new(StringArray::from(vec![Some("XNAS"), None])),
    ],
)?;
table.commit_append(arrow::batch_reader(batch.schema(), [batch]))?;

let files = table.data_files()?;
assert_eq!(files.len(), 2);
let (null_file, _) = files.iter().find(|(file, _)| file.partition[0].is_null()).unwrap();
assert!(null_file.file_path.contains("venue=null"), "the path spells it");
assert!(null_file.partition[0].is_null(), "the manifest means it");

## Reading with column pushdown

In [ ]:
use yggdryl::iceberg::{FormatVersion, PartitionSpec, Table};
use yggdryl::local::Folder;
use yggdryl::{arrow, DataType};

use arrow_array::{Int64Array, RecordBatch, RecordBatchReader, StringArray};
use std::sync::Arc;

let schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");

let path = std::env::temp_dir().join("yggdryl-docs-iceberg-pushdown");
let _ = std::fs::remove_dir_all(&path);
let mut table = Table::create(
    Folder::new(&path)?,
    FormatVersion::V2,
    schema.clone(),
    PartitionSpec::unpartitioned(),
)?;

let batch = RecordBatch::try_new(
    schema.clone().into_arrow_schema()?,
    vec![
        Arc::new(Int64Array::from(vec![1_i64, 2])),
        Arc::new(StringArray::from(vec![Some("AAPL"), Some("MSFT")])),
    ],
)?;
table.commit_append(arrow::batch_reader(batch.schema(), [batch]))?;

// The target names the columns to keep; each file's Parquet reader gets it as
// its own projection mask, so the dropped column chunk is never decoded.
let wanted = schema.without_fields(&["symbol"])?;
let reader = table.scan(Some(&wanted))?;
assert_eq!(reader.schema().fields().len(), 1);
for batch in reader {
    assert_eq!(batch?.num_columns(), 1);
}

// No target reads everything.
assert_eq!(table.scan(None)?.schema().fields().len(), 2);

## Planning a scan from the metadata

In [ ]:
use yggdryl::iceberg::{FormatVersion, PartitionSpec, Table, assign_field_ids};
use yggdryl::local::Folder;
use yggdryl::{arrow, DataType};

use arrow_array::{Int64Array, RecordBatch, StringArray};
use std::sync::Arc;

let mut schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");
assign_field_ids(&mut schema, 1)?;

let path = std::env::temp_dir().join("yggdryl-docs-iceberg-plan");
let _ = std::fs::remove_dir_all(&path);
let spec = PartitionSpec::identity(1, &schema, &["venue"])?;
let mut table = Table::create(Folder::new(&path)?, FormatVersion::V2, schema.clone(), spec)?;

// One commit per venue, so the manifest list has three rows to prune.
for (id, venue) in [(1_i64, "XNAS"), (2, "XNYS"), (3, "XLON")] {
    let batch = RecordBatch::try_new(
        schema.clone().into_arrow_schema()?,
        vec![
            Arc::new(Int64Array::from(vec![id])),
            Arc::new(StringArray::from(vec![Some(venue)])),
        ],
    )?;
    table.commit_append(arrow::batch_reader(batch.schema(), [batch]))?;
}

// Nothing is listed: the snapshot names the manifest list, whose per-partition
// summaries exclude two manifests before either Avro file is opened.
let plan = table.plan(&[("venue", "XNYS")])?;
assert_eq!(plan.tasks.len(), 1);
assert_eq!(plan.record_count()?, 1);
assert_eq!(plan.manifests_read, 1);
assert_eq!(plan.manifests_skipped(), 2);

// A filter on a column the spec does not partition on prunes on the file's
// own statistics instead, and then filters the rows the survivors hold.
let bounded = table.plan(&[("id", "3")])?;
assert_eq!(bounded.tasks.len(), 1);
assert_eq!(bounded.files_skipped(), 2);

let rows: usize = table
    .scan_where(&[("id", "3")], None)?
    .map(|batch| batch.unwrap().num_rows())
    .sum();
assert_eq!(rows, 1);

## Time travel and the inspection tables

In [ ]:
use yggdryl::iceberg::{FormatVersion, PartitionSpec, Table};
use yggdryl::local::Folder;
use yggdryl::DataType;

let root = std::env::temp_dir().join("yggdryl-doc-time-travel");
let _ = std::fs::remove_dir_all(&root);

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");
let mut table = Table::create(
    Folder::new(&root)?,
    FormatVersion::V2,
    schema.clone(),
    PartitionSpec::unpartitioned(),
)?;

let arrow_schema = schema.into_arrow_schema()?;
let one = arrow_array::RecordBatch::try_new(
    std::sync::Arc::clone(&arrow_schema),
    vec![std::sync::Arc::new(arrow_array::Int64Array::from(vec![1]))],
)?;
table.commit_append(yggdryl::arrow::batch_reader(std::sync::Arc::clone(&arrow_schema), [one]))?;
let past = table.current_snapshot().expect("one commit").snapshot_id;

let nine = arrow_array::RecordBatch::try_new(
    std::sync::Arc::clone(&arrow_schema),
    vec![std::sync::Arc::new(arrow_array::Int64Array::from(vec![9]))],
)?;
table.commit_overwrite(yggdryl::arrow::batch_reader(arrow_schema, [nine]))?;

// The present shows the overwrite; the retained snapshot shows what was.
assert_eq!(table.scan(None)?.count(), 1);
let history = table.scan_at(past, &[], None)?.next().expect("one batch")?;
assert_eq!(history.num_rows(), 1);

// Planning history prunes exactly as planning the present does.
assert_eq!(table.plan_at(past, &[])?.tasks.len(), 1);

let _ = std::fs::remove_dir_all(&root);

## Filtered reads and filtered writes

In [ ]:
use std::collections::BTreeSet;
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::iceberg::{DataFile, FormatVersion, PartitionSpec, Table, assign_field_ids};
use yggdryl::local::Folder;
use yggdryl::{arrow, DataType};

let mut schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
    DataType::Int64.nullable_field("qty"),
])?
.required_field("row");
assign_field_ids(&mut schema, 1)?;

let root = std::env::temp_dir().join("yggdryl-doc-filtered-writes");
let _ = std::fs::remove_dir_all(&root);
let spec = PartitionSpec::identity(1, &schema, &["venue"])?;
let mut table = Table::create(Folder::new(&root)?, FormatVersion::V2, schema.clone(), spec)?;

let arrow_schema = schema.into_arrow_schema()?;
let rows = |ids: Vec<i64>, venues: Vec<&'static str>, quantities: Vec<i64>| {
    let batch = RecordBatch::try_new(
        Arc::clone(&arrow_schema),
        vec![
            Arc::new(Int64Array::from(ids)),
            Arc::new(StringArray::from(venues)),
            Arc::new(Int64Array::from(quantities)),
        ],
    )
    .expect("a batch matching the root");
    arrow::batch_reader(batch.schema(), [batch])
};

// One commit per venue, so the manifest list has three rows to prune.
for (id, venue) in [(1_i64, "XNAS"), (2, "XNYS"), (3, "XLON")] {
    table.commit_append(rows(vec![id], vec![venue], vec![10]))?;
}
let inserted = table.current_snapshot().expect("three commits").snapshot_id;

// Nothing is listed and no data file is opened: the manifest list's
// per-partition summaries exclude two manifests before either is read.
let plan = table.plan(&[("venue", "XNYS")])?;
assert_eq!(plan.tasks.len(), 1);
assert_eq!(plan.record_count()?, 1);
assert_eq!(plan.manifests_read, 1);
assert_eq!(plan.manifests_skipped(), 2);
assert_eq!(table.scan_where(&[("venue", "XNYS")], None)?.count(), 1);

// A filter on a column the spec does not partition on prunes on the file's
// own recorded bounds instead, then filters the rows the survivors hold.
assert_eq!(table.plan(&[("id", "3")])?.files_skipped(), 2);

// A filtered overwrite replaces the files the filter selects and carries
// every other file into the new snapshot at its own path, statistics and all.
let paths = |files: Vec<(DataFile, PartitionSpec)>| -> BTreeSet<String> {
    files.into_iter().map(|(file, _)| file.file_path.to_string()).collect()
};
let before = paths(table.data_files()?);
table.commit_overwrite_where(&[("venue", "XNYS")], rows(vec![2], vec!["XNYS"], vec![99]))?;
let after = paths(table.data_files()?);
assert_eq!(before.difference(&after).count(), 1, "one partition was rewritten");
assert_eq!(before.intersection(&after).count(), 2, "the others were carried");

// A merge upserts on the key: 3 is stored and updates, 4 is new and appends.
table.commit_merge(rows(vec![3, 4], vec!["XLON", "XLON"], vec![7, 8]), &["id".to_owned()], true)?;
let total: usize = table
    .scan(None)?
    .map(|batch| batch.map(|batch| batch.num_rows()))
    .sum::<Result<usize, _>>()?;
assert_eq!(total, 4);

// Narrowed first: a merge into one partition can read no other partition.
table.commit_merge_where(
    &[("venue", "XNAS")],
    rows(vec![1], vec!["XNAS"], vec![42]),
    &["id".to_owned()],
    true,
)?;

// History plans the same way: the snapshot before the overwrite still
// selects one file for that partition.
assert_eq!(table.plan_at(inserted, &[("venue", "XNYS")])?.tasks.len(), 1);

let _ = std::fs::remove_dir_all(&root);

## The three record methods over a table

In [ ]:
use yggdryl::generic::IORecordOptions;
use yggdryl::iceberg::{FormatVersion, PartitionSpec, Table, assign_field_ids};
use yggdryl::io::{IOBase, IOMedia};
use yggdryl::local::Folder;
use yggdryl::{arrow, DataType};

use arrow_array::{Int64Array, RecordBatch, StringArray};
use std::sync::Arc;

let mut schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");
assign_field_ids(&mut schema, 1)?;

let path = std::env::temp_dir().join("yggdryl-docs-iceberg-records");
let _ = std::fs::remove_dir_all(&path);
let spec = PartitionSpec::identity(1, &schema, &["venue"])?;
Table::create(Folder::new(&path)?, FormatVersion::V2, schema.clone(), spec)?;

let arrow_schema = schema.into_arrow_schema()?;
let rows = |ids: Vec<i64>, venues: Vec<&'static str>| {
    let batch = RecordBatch::try_new(
        Arc::clone(&arrow_schema),
        vec![
            Arc::new(Int64Array::from(ids)),
            Arc::new(StringArray::from(venues)),
        ],
    )
    .expect("a batch matching the root");
    arrow::batch_reader(batch.schema(), [batch])
};

// The folder *is* the table, so the ordinary record surface reaches it. Its
// options come from the metadata, before a single data file exists.
let mut folder = Folder::new(&path)?;
let options = folder.record_options()?;
folder.overwrite_arrow_reader(rows(vec![1, 2], vec!["XNAS", "XNYS"]), &options)?;
folder.append_arrow_reader(rows(vec![3], vec!["XLON"]), &options)?;

// A match key upserts: `2` is stored and updates, `9` is new and appends.
let merging = options.clone().with_merge_by_names(["id"]);
folder.merge_arrow_reader(rows(vec![2, 9], vec!["XNYS", "XLON"]), &merging)?;

let total: usize = folder
    .read_arrow_reader(&options)?
    .map(|batch| batch.unwrap().num_rows())
    .sum();
assert_eq!(total, 4);

// Each call was one commit, and the read went through the last one.
let table = Table::open(Folder::new(&path)?)?;
assert_eq!(table.metadata().snapshots().len(), 3);

In [ ]:
use yggdryl::generic::IORecordOptions;
use yggdryl::iceberg::{FormatVersion, PartitionSpec, Table, assign_field_ids};
use yggdryl::io::{IOBase, IOMedia};
use yggdryl::local::Folder;
use yggdryl::{arrow, DataType, MimeType};

use arrow_array::{Int64Array, RecordBatch, StringArray};
use std::sync::Arc;

let mut schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");
assign_field_ids(&mut schema, 1)?;

let path = std::env::temp_dir().join("yggdryl-docs-iceberg-table-handle");
let _ = std::fs::remove_dir_all(&path);
let spec = PartitionSpec::identity(1, &schema, &["venue"])?;
let mut table = Table::create(Folder::new(&path)?, FormatVersion::V2, schema.clone(), spec)?;

// The role is the table's own, and it costs nothing to say so.
assert_eq!(IOBase::kind(&table), yggdryl::IOKind::Table);
assert!(table.is_container());
assert!(table.is_tabular());
assert!(!table.is_atomic());

// The record surface answers before a single data file exists: the encoding
// from the metadata, the schema with its field identifiers.
let options = table.record_options()?;
assert_eq!(options.mime_type(), MimeType::PARQUET);
assert_eq!(
    table.read_arrow_field(&options)?.fields()[0].parquet_field_id()?,
    Some(1),
);

let arrow_schema = schema.into_arrow_schema()?;
let rows = |ids: Vec<i64>, venues: Vec<&'static str>| {
    let batch = RecordBatch::try_new(
        Arc::clone(&arrow_schema),
        vec![
            Arc::new(Int64Array::from(ids)),
            Arc::new(StringArray::from(venues)),
        ],
    )
    .expect("a batch matching the root");
    arrow::batch_reader(batch.schema(), [batch])
};

// Each generic write is one commit, and the value's metadata follows it
// without reopening anything.
table.append_arrow_reader(rows(vec![1, 2], vec!["XNAS", "XNYS"]), &options)?;
let merging = options.clone().with_merge_by_names(["id"]);
table.merge_arrow_reader(rows(vec![2, 9], vec!["XNYS", "XLON"]), &merging)?;
assert_eq!(table.metadata().snapshots().len(), 2);
assert_eq!(table.current_snapshot().unwrap().operation(), "overwrite");

// A partition filter is answered by the scan plan, so the other partitions'
// files are never opened.
let filtered = options.clone().with_filter_partitions([("venue", "XNYS")]);
let matching: usize = table
    .read_arrow_reader(&filtered)?
    .map(|batch| batch.unwrap().num_rows())
    .sum();
assert_eq!(matching, 1);

In [ ]:
use yggdryl::generic::IORecordOptions;
use yggdryl::iceberg::{FormatVersion, PartitionSpec, Table, assign_field_ids};
use yggdryl::io::{IOBase, IOMedia};
use yggdryl::local::Folder;
use yggdryl::{arrow, DataType};

use arrow_array::{Int64Array, RecordBatch, StringArray};
use std::sync::Arc;

let mut schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row");
assign_field_ids(&mut schema, 1)?;

let path = std::env::temp_dir().join("yggdryl-docs-iceberg-partition");
let _ = std::fs::remove_dir_all(&path);
let spec = PartitionSpec::identity(1, &schema, &["venue"])?;
let mut table = Table::create(Folder::new(&path)?, FormatVersion::V2, schema.clone(), spec)?;

let batch = RecordBatch::try_new(
    schema.into_arrow_schema()?,
    vec![
        Arc::new(Int64Array::from(vec![1_i64, 2])),
        Arc::new(StringArray::from(vec![Some("XNAS"), Some("XNYS")])),
    ],
)?;
table.commit_append(arrow::batch_reader(batch.schema(), [batch]))?;

let partition = Folder::new(path.join("data").join("venue=XNYS"))?;
let options = partition.record_options()?;
let rows: usize = partition
    .read_arrow_reader(&options)?
    .map(|batch| batch.unwrap().num_rows())
    .sum();
assert_eq!(rows, 1);

## A warehouse of tables

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch, StringArray};
use yggdryl::iceberg::Catalog;
use yggdryl::local::Folder;
use yggdryl::DataType;

let warehouse = std::env::temp_dir().join("yggdryl-doc-warehouse");
let _ = std::fs::remove_dir_all(&warehouse);
let catalog = Catalog::new(Folder::new(&warehouse)?);

// Rows and a name are enough: the first append creates the table with the
// schema the rows carry, and the second appends to it.
let schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    DataType::Utf8.nullable_field("venue"),
])?
.required_field("row")
.with_partition_fields(&["venue"])?;
let arrow_schema = schema.into_arrow_schema()?;
let rows = |ids: &[i64], venues: &[&str]| {
    RecordBatch::try_new(
        Arc::clone(&arrow_schema),
        vec![
            Arc::new(Int64Array::from(ids.to_vec())),
            Arc::new(StringArray::from(venues.to_vec())),
        ],
    )
};
let first = rows(&[1, 2], &["XNAS", "XNYS"])?;
let table = catalog
    .tables()
    .append_arrow_reader("nyc.trades", yggdryl::arrow::batch_reader(first.schema(), [first]))?;
let rows_read: usize = table.scan(None)?.map(|batch| batch.map(|b| b.num_rows())).sum::<Result<usize, _>>()?;
assert_eq!(rows_read, 2);

let second = rows(&[3], &["XNAS"])?;
catalog
    .tables()
    .append_arrow_reader("nyc.trades", yggdryl::arrow::batch_reader(second.schema(), [second]))?;

// The partition marks the schema carried became the table's spec.
let reopened = catalog.table("nyc.trades")?;
assert_eq!(reopened.metadata().default_spec()?.fields[0].name, "venue");
assert!(catalog.tables().contains("nyc.trades")?);
let namespaces: Vec<String> =
    catalog.namespaces().iter().collect::<yggdryl::Result<_>>()?;
assert_eq!(namespaces, ["nyc"]);
let tables: Vec<String> = catalog
    .namespaces()
    .get("nyc")?
    .tables()
    .iter()
    .collect::<yggdryl::Result<_>>()?;
assert_eq!(tables, ["trades"]);

let _ = std::fs::remove_dir_all(&warehouse);

### The object model: namespaces of tables

In [ ]:
use yggdryl::iceberg::Catalog;
use yggdryl::local::Folder;

let root = std::env::temp_dir().join("yggdryl-doc-views");
let _ = std::fs::remove_dir_all(&root);
let catalog = Catalog::new(Folder::new(&root)?);

// Constructing the views touches nothing; every answer is storage's.
let namespaces = catalog.namespaces();
assert_eq!(namespaces.iter().count(), 0);
let sales = namespaces.open_or_create("sales")?;
assert!(!sales.tables().contains("orders")?);
assert!(namespaces.contains("sales")?);

// The namespace document is what makes the empty namespace durable, and
// it is where its properties live.
sales.update_properties([("region".to_owned(), "eu".to_owned())], [])?;
assert_eq!(
    sales.properties()?.get("region").map(String::from),
    Some("eu".to_owned())
);

let _ = std::fs::remove_dir_all(&root);

## Data files aim at a size

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::iceberg::{Catalog, FormatVersion};
use yggdryl::local::Folder;
use yggdryl::DataType;

let warehouse = std::env::temp_dir().join("yggdryl-doc-compaction");
let _ = std::fs::remove_dir_all(&warehouse);
let catalog = Catalog::new(Folder::new(&warehouse)?);

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");
let arrow_schema = schema.clone().into_arrow_schema()?;
let one = |id: i64| {
    RecordBatch::try_new(
        Arc::clone(&arrow_schema),
        vec![Arc::new(Int64Array::from(vec![id]))],
    )
};

// Five appends, five snapshots, five small files.
let mut table = catalog.tables().create("tiny.rows", schema)?;
for id in 0..5 {
    let batch = one(id)?;
    table.commit_append(yggdryl::arrow::batch_reader(batch.schema(), [batch]))?;
}
assert_eq!(table.inspect_files()?.next().expect("one batch")?.num_rows(), 5);

// Compaction rewrites the small groups as one replace commit and reports it.
let compaction = table.compact()?;
assert_eq!(compaction.files_before, 5);
assert_eq!(compaction.files_after, 1);
assert_eq!(table.scan(None)?.map(|batch| batch.map(|b| b.num_rows())).sum::<Result<usize, _>>()?, 5);

// Nothing to do is a no-op that commits nothing.
assert_eq!(table.compact()?, yggdryl::iceberg::Compaction::default());

let _ = std::fs::remove_dir_all(&warehouse);

## One options value, three layers

In [ ]:
use yggdryl::iceberg::{
    FormatVersion, IcebergOptions, PartitionSpec, Table,
};
use yggdryl::local::Folder;
use yggdryl::DataType;

let root = std::env::temp_dir().join("yggdryl-doc-options");
let _ = std::fs::remove_dir_all(&root);

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");
let mut table = Table::create(
    Folder::new(&root)?,
    FormatVersion::V2,
    schema,
    PartitionSpec::unpartitioned(),
)?;

// Nothing set: every field answers its documented default.
assert_eq!(table.options()?.commit_retries(), 4);
assert_eq!(table.options()?.commit_total_timeout_ms(), 1_800_000);
assert_eq!(table.options()?.target_file_size_bytes(), 512 * 1024 * 1024);

// The property layer is the table's own metadata, one commit away.
table.commit_metadata_changes(|metadata| {
    metadata.set_property(IcebergOptions::COMMIT_RETRIES_KEY, "9")?;
    Ok(())
})?;
assert_eq!(table.options()?.commit_retries(), 9);

// An explicit override shadows the property on this handle alone;
// nothing is written, and an unset field still resolves the other layers.
table.set_options(IcebergOptions::new().with_commit_retries(2));
assert_eq!(table.options()?.commit_retries(), 2);
assert_eq!(table.options()?.commit_min_backoff_ms(), 100);

let _ = std::fs::remove_dir_all(&root);

## The data-file MIME type

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::iceberg::{FormatVersion, IcebergOptions, PartitionSpec, Table};
use yggdryl::local::Folder;
use yggdryl::{DataType, MimeType};

let root = std::env::temp_dir().join("yggdryl-doc-data-format");
let _ = std::fs::remove_dir_all(&root);

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");
let mut table = Table::create(
    Folder::new(&root)?,
    FormatVersion::V2,
    schema.clone(),
    PartitionSpec::unpartitioned(),
)?;

let batch = RecordBatch::try_new(
    schema.into_arrow_schema()?,
    vec![Arc::new(Int64Array::from(vec![1_i64]))],
)?;

// One Parquet append, then one Avro append via the explicit option.
table.commit_append(yggdryl::arrow::batch_reader(batch.schema(), [batch.clone()]))?;
table.set_options(
    IcebergOptions::new().try_with_data_mime_type(MimeType::AVRO)?,
);
table.commit_append(yggdryl::arrow::batch_reader(batch.schema(), [batch]))?;

// The manifest records what was written, and the mixed table scans whole.
let mut formats: Vec<MimeType> = table
    .data_files()?
    .into_iter()
    .map(|(file, _)| file.mime_type)
    .collect();
formats.sort();
assert_eq!(formats, [MimeType::AVRO, MimeType::PARQUET]);
assert_eq!(table.scan(None)?.count(), 2);

let _ = std::fs::remove_dir_all(&root);

## Concurrent writers and commit retries

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::iceberg::{FormatVersion, PartitionSpec, Table};
use yggdryl::local::Folder;
use yggdryl::DataType;

let root = std::env::temp_dir().join("yggdryl-doc-concurrency");
let _ = std::fs::remove_dir_all(&root);

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");
Table::create(
    Folder::new(&root)?,
    FormatVersion::V2,
    schema.clone(),
    PartitionSpec::unpartitioned(),
)?;

// Two handles opened at the same version, each unaware of the other.
let mut left = Table::open(Folder::new(&root)?)?;
let mut right = Table::open(Folder::new(&root)?)?;

let arrow_schema = schema.into_arrow_schema()?;
let one = |id: i64| {
    RecordBatch::try_new(
        Arc::clone(&arrow_schema),
        vec![Arc::new(Int64Array::from(vec![id]))],
    )
};

let batch = one(1)?;
left.commit_append(yggdryl::arrow::batch_reader(batch.schema(), [batch]))?;

// The right handle is now stale; its commit observes the winner,
// rebases onto it, and lands as the next version.
let batch = one(2)?;
right.commit_append(yggdryl::arrow::batch_reader(batch.schema(), [batch]))?;

// Both rows survive, on one line of history, and the rebased handle
// is current: no re-open needed to see the winner's row.
let rows: usize = right
    .scan(None)?
    .map(|batch| batch.map(|b| b.num_rows()))
    .sum::<Result<usize, _>>()?;
assert_eq!(rows, 2);
assert_eq!(right.inspect_history()?.next().expect("one batch")?.num_rows(), 2);

let _ = std::fs::remove_dir_all(&root);

## Branches and tags

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::iceberg::{FormatVersion, PartitionSpec, Table};
use yggdryl::local::Folder;
use yggdryl::DataType;

let root = std::env::temp_dir().join("yggdryl-doc-branching");
let _ = std::fs::remove_dir_all(&root);

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");
let mut table = Table::create(
    Folder::new(&root)?,
    FormatVersion::V2,
    schema.clone(),
    PartitionSpec::unpartitioned(),
)?;

let arrow_schema = schema.into_arrow_schema()?;
let one = |id: i64| {
    RecordBatch::try_new(
        Arc::clone(&arrow_schema),
        vec![Arc::new(Int64Array::from(vec![id]))],
    )
};

let batch = one(1)?;
table.commit_append(yggdryl::arrow::batch_reader(batch.schema(), [batch]))?;
let audited = table.current_snapshot().expect("one commit").snapshot_id;

// The tag pins the audited state; the table keeps moving.
table.create_tag("audit-2026", audited)?;
let batch = one(2)?;
table.commit_append(yggdryl::arrow::batch_reader(batch.schema(), [batch]))?;
table.create_branch("review", audited)?;

// Every ref reads as the complete table it names.
assert_eq!(table.scan_ref("audit-2026", &[], None)?.count(), 1);
assert_eq!(table.scan_ref("review", &[], None)?.count(), 1);
assert_eq!(table.scan(None)?.count(), 2);

// A branch fast-forwards only along its own ancestry: the target must
// reach the branch's head by parent ids, so no history can be lost.
let head = table.current_snapshot().expect("two commits").snapshot_id;
table.fast_forward_branch("review", head)?;
assert_eq!(table.snapshot_by_ref("review")?.snapshot_id, head);

// Removing a ref removes the name; the snapshots stay retained.
let removed = table.remove_snapshot_ref("review")?;
assert_eq!(removed.snapshot_id, head);

// Expiry honors every ref's retention: the tagged snapshot survives
// a cutoff that would otherwise expire everything old.
assert!(table
    .expire_snapshots(Some(i64::MAX), None, &[])?
    .is_empty());

let _ = std::fs::remove_dir_all(&root);

## Reading many files at once

In [ ]:
use std::sync::Arc;

use arrow_array::{Int64Array, RecordBatch};
use yggdryl::iceberg::{
    FormatVersion, IcebergOptions, PartitionSpec, Table,
};
use yggdryl::local::Folder;
use yggdryl::DataType;

let root = std::env::temp_dir().join("yggdryl-doc-parallel-read");
let _ = std::fs::remove_dir_all(&root);

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");
let mut table = Table::create(
    Folder::new(&root)?,
    FormatVersion::V2,
    schema.clone(),
    PartitionSpec::unpartitioned(),
)?;

let arrow_schema = schema.into_arrow_schema()?;
for id in 0..3 {
    let batch = RecordBatch::try_new(
        Arc::clone(&arrow_schema),
        vec![Arc::new(Int64Array::from(vec![id]))],
    )?;
    table.commit_append(yggdryl::arrow::batch_reader(batch.schema(), [batch]))?;
}

// Three tiny files sit below both thresholds, so this table reads
// sequentially by default; forcing the thresholds down demonstrates
// that the fan-out changes nothing the caller can observe.
let sequential: Vec<RecordBatch> = table.scan(None)?.collect::<Result<_, _>>()?;
table.set_options(
    IcebergOptions::new()
        .try_with_read_parallelism(2)?
        .with_read_parallel_min_files(1)
        .with_read_parallel_min_file_size_bytes(0),
);
let fanned: Vec<RecordBatch> = table.scan(None)?.collect::<Result<_, _>>()?;
assert_eq!(sequential, fanned);

let _ = std::fs::remove_dir_all(&root);

## The Spark quickstart, locally

In [ ]:
use std::sync::Arc;

use arrow_array::{Float32Array, Float64Array, Int64Array, RecordBatch, StringArray};
use yggdryl::generic::Holder;
use yggdryl::iceberg::Table;
use yggdryl::local::Folder;
use yggdryl::DataType;

let root = std::env::temp_dir().join("yggdryl-doc-nyc-taxis");
let _ = std::fs::remove_dir_all(&root);
let catalog = yggdryl::iceberg::Catalog::new(Folder::new(&root)?);

// CREATE TABLE nyc.taxis (...) PARTITIONED BY (vendor_id)
// The partition mark on the schema is the whole PARTITIONED BY clause.
let schema = DataType::from_fields([
    DataType::Int64.required_field("vendor_id"),
    DataType::Int64.required_field("trip_id"),
    DataType::Float32.nullable_field("trip_distance"),
    DataType::Float64.nullable_field("fare_amount"),
    DataType::Utf8.nullable_field("store_and_fwd_flag"),
])?
.required_field("row")
.with_partition_fields(&["vendor_id"])?;
let mut table = catalog.tables().create("nyc.taxis", schema.clone())?;
let schema = table.schema()?.clone();

// INSERT INTO nyc.taxis VALUES (...)
let arrow_schema = schema.into_arrow_schema()?;
let taxis = |vendors: &[i64], trips: &[i64], distances: &[f32], fares: &[f64], flags: &[&str]| {
    RecordBatch::try_new(
        Arc::clone(&arrow_schema),
        vec![
            Arc::new(Int64Array::from(vendors.to_vec())),
            Arc::new(Int64Array::from(trips.to_vec())),
            Arc::new(Float32Array::from(distances.to_vec())),
            Arc::new(Float64Array::from(fares.to_vec())),
            Arc::new(StringArray::from(flags.to_vec())),
        ],
    )
};
let rows = taxis(
    &[1, 2, 2, 1],
    &[1_000_371, 1_000_372, 1_000_373, 1_000_374],
    &[1.8, 2.5, 0.9, 8.4],
    &[15.32, 22.15, 9.01, 42.13],
    &["N", "N", "N", "Y"],
)?;
table.commit_append(yggdryl::arrow::batch_reader(rows.schema(), [rows]))?;

// SELECT * FROM nyc.taxis
let fares = |table: &Table<Holder>| -> Result<Vec<(i64, f64)>, Box<dyn std::error::Error>> {
    let mut rows = Vec::new();
    for batch in table.scan(None)? {
        let batch = batch?;
        let trips = batch.column_by_name("trip_id").expect("the trip column");
        let fares = batch.column_by_name("fare_amount").expect("the fare column");
        let trips = trips.as_any().downcast_ref::<Int64Array>().expect("int64");
        let fares = fares.as_any().downcast_ref::<Float64Array>().expect("float64");
        for row in 0..batch.num_rows() {
            rows.push((trips.value(row), fares.value(row)));
        }
    }
    rows.sort_by_key(|(trip, _)| *trip);
    Ok(rows)
};
assert_eq!(fares(&table)?.len(), 4);
assert_eq!(fares(&table)?[0], (1_000_371, 15.32));
let before_changes = table.current_snapshot().expect("the insert").snapshot_id;

// UPDATE nyc.taxis SET fare_amount = 16.32 WHERE trip_id = 1000371
// An update is a merge: the incoming row matches on the key and replaces.
let update = taxis(&[1], &[1_000_371], &[1.8], &[16.32], &["N"])?;
table.commit_merge(
    yggdryl::arrow::batch_reader(update.schema(), [update]),
    &["trip_id".to_owned()],
    true,
)?;
assert_eq!(fares(&table)?[0], (1_000_371, 16.32));
assert_eq!(fares(&table)?.len(), 4);

// DELETE FROM nyc.taxis WHERE vendor_id = 1
// A delete is a filtered overwrite with nothing incoming: the selected
// partition is replaced by no rows, and every other file is carried over.
table.commit_overwrite_where(
    &[("vendor_id", "1")],
    yggdryl::arrow::batch_reader(Arc::clone(&arrow_schema), []),
)?;
assert_eq!(
    fares(&table)?,
    [(1_000_372, 22.15), (1_000_373, 9.01)],
);

// ALTER TABLE nyc.taxis ADD COLUMN fare_per_distance float
let mut update = yggdryl::iceberg::SchemaUpdate::from_metadata(table.metadata())?;
update.add_column("", DataType::Float32.nullable_field("fare_per_distance"));
let evolved = update.into_field()?;
table.commit_metadata_changes(|metadata| {
    // The new column got the next unused id; a retired id is never reused.
    let schema_id = metadata.add_schema(evolved.clone())?;
    metadata.set_current_schema(schema_id)
})?;
let widened = table.scan(None)?.next().expect("one batch")?;
assert_eq!(widened.schema().fields().len(), 6);
assert_eq!(widened.column_by_name("fare_per_distance").expect("the new column").null_count(), 2);

// Time travel: the table before the update and the delete is still there.
assert_eq!(table.scan_at(before_changes, &[], None)?.map(|batch| batch.map(|b| b.num_rows())).sum::<Result<usize, _>>()?, 4);

// SELECT * FROM nyc.taxis.history / .snapshots / .files
let history = table.inspect_history()?.next().expect("one batch")?;
assert_eq!(history.num_rows(), 3);
let files = table.inspect_files()?.next().expect("one batch")?;
assert_eq!(files.num_rows(), 1);

let _ = std::fs::remove_dir_all(&root);

## Schema evolution and field ids

In [ ]:
use yggdryl::iceberg::{FormatVersion, PartitionSpec, SchemaUpdate, Table};
use yggdryl::local::Folder;
use yggdryl::{arrow, DataType};

use arrow_array::{Int64Array, RecordBatch};
use std::sync::Arc;

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");

let path = std::env::temp_dir().join("yggdryl-docs-iceberg-evolution");
let _ = std::fs::remove_dir_all(&path);
let mut table = Table::create(
    Folder::new(&path)?,
    FormatVersion::V2,
    schema.clone(),
    PartitionSpec::unpartitioned(),
)?;

let batch = RecordBatch::try_new(
    schema.into_arrow_schema()?,
    vec![Arc::new(Int64Array::from(vec![1_i64]))],
)?;
table.commit_append(arrow::batch_reader(batch.schema(), [batch]))?;

// Add a column. Numbering continues above `last-column-id`, so the new column
// can never be confused with a dropped one.
let mut update = SchemaUpdate::from_metadata(table.metadata())?;
update.add_column("", DataType::Int64.nullable_field("quantity"));
let evolved = update.into_field()?;
assert_eq!(table.evolve_schema(evolved)?, 1, "the new schema's id");

// The old schema is retained, so the snapshot written under it still reads.
assert_eq!(table.metadata().schemas().len(), 2);
assert_eq!(table.metadata().schema_by_id(0).unwrap().field_len(), 1);

// And the file written before the column existed reads it as null.
for batch in table.scan(None)? {
    let batch = batch?;
    assert_eq!(batch.num_columns(), 2);
    assert_eq!(batch.column_by_name("quantity").unwrap().null_count(), batch.num_rows());
}

In [ ]:
use yggdryl::iceberg::{assign_field_ids, last_column_id};
use yggdryl::DataType;

let leg = DataType::from_fields([DataType::decimal(18, 4)?.required_field("price")])?;
let mut schema = DataType::from_fields([
    DataType::Int64.required_field("id"),
    leg.nullable_field("leg"),
])?
.required_field("row");

// Depth first from `start`; the return value is the first id it did not use.
assert_eq!(assign_field_ids(&mut schema, 1)?, 4);
assert_eq!(schema.fields()[0].parquet_field_id()?, Some(1));
assert_eq!(schema.fields()[1].parquet_field_id()?, Some(2));
assert_eq!(schema.fields()[1].fields()[0].parquet_field_id()?, Some(3));
assert_eq!(last_column_id(&schema)?, 3, "what a table records as last-column-id");

// The root is not a column, so it is not numbered.
assert_eq!(schema.parquet_field_id()?, None);

// A field that already carries an id keeps it, so a second pass changes nothing.
assert_eq!(assign_field_ids(&mut schema, 100)?, 100);
assert_eq!(schema.fields()[0].parquet_field_id()?, Some(1));

In [ ]:
use yggdryl::iceberg::schema_into_json;
use yggdryl::DataType;

let schema = DataType::from_fields([DataType::Int64.required_field("id")])?
    .required_field("row");

let message = schema_into_json(&schema).unwrap_err().to_string();
assert!(message.contains("assign_field_ids"));

## Evolving a schema

In [ ]:
use yggdryl::iceberg::{can_promote, FormatVersion, PartitionSpec, SchemaUpdate, Table};
use yggdryl::local::Folder;
use yggdryl::DataType;

let root = std::env::temp_dir().join("yggdryl-doc-evolution");
let _ = std::fs::remove_dir_all(&root);
let schema = DataType::from_fields([
    DataType::Int32.required_field("id"),
    DataType::Utf8.nullable_field("symbol"),
])?
.required_field("row");
let mut table = Table::create(
    Folder::new(&root)?,
    FormatVersion::V2,
    schema,
    PartitionSpec::unpartitioned(),
)?;

// Legal promotions pass; anything else is refused naming both sides.
assert!(can_promote(&DataType::Int32, &DataType::Int64).is_ok());
assert!(can_promote(&DataType::decimal(10, 2)?, &DataType::decimal(18, 2)?).is_ok());
let message = can_promote(&DataType::Int64, &DataType::Int32).unwrap_err().to_string();
assert!(message.contains("int64") && message.contains("int32"));

// Widen id, rename symbol, add venue - one evolved schema, one commit.
let mut update = SchemaUpdate::from_metadata(table.metadata())?;
update.update_type("id", DataType::Int64);
update.rename_column("symbol", "ticker");
update.add_column("", DataType::Utf8.nullable_field("venue"));
let evolved = update.into_field()?;

table.commit_metadata_changes(|metadata| {
    let schema_id = metadata.add_schema(evolved.clone())?;
    metadata.set_current_schema(schema_id)
})?;

let current = table.schema()?;
assert_eq!(current.get_field_by_name("id").expect("the column").data_type(), &DataType::Int64);
// A renamed column keeps its identifier: the name is a label, the id is the column.
assert_eq!(current.get_field_by_name("ticker").expect("the column").parquet_field_id()?, Some(2));
assert_eq!(current.get_field_by_name("venue").expect("the column").parquet_field_id()?, Some(3));

let _ = std::fs::remove_dir_all(&root);

## Schemas as documents

In [ ]:
use yggdryl::iceberg::{schema_from_json, schema_into_json};
use yggdryl::{json, DataType};

let document = json::from_utf8(
    r#"{"type":"struct","schema-id":0,"fields":[
        {"id":1,"name":"id","required":true,"type":"long"},
        {"id":2,"name":"symbol","required":false,"type":"string"}
    ]}"#,
)?;

// An Iceberg schema is a non-null struct field; its columns are the children.
let schema = schema_from_json("row", &document)?;
assert!(schema.is_struct());
assert!(!schema.is_nullable());
assert_eq!(schema.field_len(), 2);
assert_eq!(schema.fields()[0].data_type(), &DataType::Int64);

// `required` inverts into nullability, and `id` becomes PARQUET:field_id.
assert!(!schema.fields()[0].is_nullable());
assert!(schema.fields()[1].is_nullable());
assert_eq!(schema.fields()[0].parquet_field_id()?, Some(1));
assert_eq!(schema.fields()[0].get_metadata("PARQUET:field_id"), Some("1"));

// The same document comes back out.
assert_eq!(schema_into_json(&schema)?, document);

## Primitive types

In [ ]:
use yggdryl::iceberg::PrimitiveType;
use yggdryl::{DataType, TimeUnit};

// Every Iceberg primitive name has exactly one physical datatype.
assert_eq!(PrimitiveType::from_str("long")?.into_data_type()?, DataType::Int64);
assert_eq!(PrimitiveType::from_str("string")?.into_data_type()?, DataType::Utf8);
assert_eq!(
    PrimitiveType::from_str("decimal(18, 4)")?.into_data_type()?,
    DataType::decimal(18, 4)?
);

// Iceberg fixed every temporal resolution at microseconds until v3 added the
// nanosecond pair.
assert_eq!(
    PrimitiveType::from_str("timestamp")?.into_data_type()?,
    DataType::Timestamp(TimeUnit::Microsecond, None)
);
assert_eq!(
    PrimitiveType::from_str("timestamp_ns")?.into_data_type()?,
    DataType::Timestamp(TimeUnit::Nanosecond, None)
);
assert_eq!(
    PrimitiveType::from_str("time")?.into_data_type()?,
    DataType::time(TimeUnit::Microsecond)?
);

// A v3 `unknown` column always reads as null, which Arrow spells exactly.
assert_eq!(PrimitiveType::from_str("unknown")?.into_data_type()?, DataType::Null);

// A name round trips through `Display`.
assert_eq!(PrimitiveType::from_str("fixed[16]")?.to_string(), "fixed[16]");

In [ ]:
use yggdryl::iceberg::PrimitiveType;
use yggdryl::DataType;

// The variants that differ only in physical layout collapse onto one name.
assert_eq!(PrimitiveType::from_data_type(&DataType::Utf8)?, PrimitiveType::String);
assert_eq!(PrimitiveType::from_data_type(&DataType::LargeUtf8)?, PrimitiveType::String);
assert_eq!(PrimitiveType::from_data_type(&DataType::BinaryView)?, PrimitiveType::Binary);
assert_eq!(
    PrimitiveType::from_data_type(&DataType::decimal64(9, 2)?)?,
    PrimitiveType::Decimal { precision: 9, scale: 2 }
);

// A datatype Iceberg cannot express is reported, never approximated.
let message = PrimitiveType::from_data_type(&DataType::Int8).unwrap_err().to_string();
assert!(message.contains("int8"));
assert!(PrimitiveType::from_data_type(&DataType::Int16).is_err());

// A UUID is 16 bytes on the wire and nothing more, so it writes back as `fixed[16]`.
assert_eq!(
    PrimitiveType::from_data_type(&PrimitiveType::Uuid.into_data_type()?)?.to_string(),
    "fixed[16]"
);

In [ ]:
use yggdryl::iceberg::PrimitiveType;
use yggdryl::{DataType, Scheme};

// The narrow integers widen; the refusals stay refusals.
let widened = DataType::Int8.into_scheme_compat(&Scheme::ICEBERG)?;
assert_eq!(widened, DataType::Int32);
assert_eq!(PrimitiveType::from_data_type(&widened)?.to_string(), "int");
assert!(DataType::Interval(yggdryl::TimeUnit::YearMonth).into_scheme_compat(&Scheme::ICEBERG).is_err());

## Nested types

In [ ]:
use yggdryl::iceberg::{schema_from_json, schema_into_json};
use yggdryl::{json, DataType};

let document = json::from_utf8(
    r#"{"type":"struct","schema-id":0,"fields":[
        {"id":1,"name":"legs","required":false,"type":{
            "type":"list","element-id":2,"element":{
                "type":"struct","fields":[
                    {"id":3,"name":"price","required":true,"type":"decimal(18, 4)"}
                ]
            },"element-required":true
        }},
        {"id":4,"name":"tags","required":false,"type":{
            "type":"map","key-id":5,"key":"string","value-id":6,"value":"int",
            "value-required":false
        }}
    ]}"#,
)?;

let schema = schema_from_json("row", &document)?;

// A list becomes a `List` whose item field is named `element` and carries `element-id`.
let legs = &schema.fields()[0];
let DataType::List(element) = legs.data_type() else { panic!("expected a list") };
assert_eq!(element.name(), "element");
assert_eq!(element.parquet_field_id()?, Some(2));
assert!(!element.is_nullable());
assert_eq!(element.fields()[0].name(), "price");

// A map becomes a `Map` over a non-null `entries` struct of `key` and `value`.
let tags = &schema.fields()[1];
let DataType::Map(map) = tags.data_type() else { panic!("expected a map") };
assert_eq!(map.entries().name(), "entries");
assert!(!map.entries().is_nullable());
assert!(!map.entries().fields()[0].is_nullable());
assert!(map.entries().fields()[1].is_nullable());
assert_eq!(map.entries().fields()[0].parquet_field_id()?, Some(5));

assert_eq!(schema_into_json(&schema)?, document);

## Into a data file

In [ ]:
use arrow_array::RecordBatch;
use yggdryl::arrow;
use yggdryl::iceberg::schema_from_json;
use yggdryl::io::{Buffer, IOMedia};
use yggdryl::json;
use yggdryl::parquet::Parquet;

let document = json::from_utf8(
    r#"{"type":"struct","fields":[
        {"id":7,"name":"id","required":true,"type":"long"},
        {"id":8,"name":"symbol","required":false,"type":"string"}
    ]}"#,
)?;
let schema = schema_from_json("row", &document)?;

let mut media = Parquet::new(Buffer::new());
let options = media.record_options()?;
media.overwrite_arrow_reader(
    arrow::batch_reader(
        schema.into_arrow_schema()?,
        std::iter::empty::<RecordBatch>(),
    ),
    &options,
)?;

// The ids Iceberg assigned are the ids in the file.
let written = media.read_arrow_field(&options)?;
assert_eq!(written.fields()[0].parquet_field_id()?, Some(7));
assert_eq!(written.fields()[1].parquet_field_id()?, Some(8));
assert!(!written.fields()[0].is_nullable());